In [0]:
%pip install openpyxl xlsxwriter
print("✅ Paquetes openpyxl y xlsxwriter instalados correctamente.")

In [0]:
# =============================================================
# CELDA 1: SETUP — Imports, Credenciales, Config S3
# =============================================================
import json
import boto3
import unicodedata
import re
import os
import sys
import importlib
import importlib.util
from functools import reduce
from pyspark.sql.functions import (
    col, coalesce, regexp_extract, trim, lit,
    when, length, lower, row_number, desc, current_timestamp,
    regexp_replace
    )
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Credenciales: Databricks secrets (producción) → archivo local (desarrollo)
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
    }
    print("✅ Credenciales desde Databricks Secrets.")
except Exception:
    try:
        with open("aws_secrets.json", "r") as f:
            config = json.load(f)
        print("✅ Credenciales desde aws_secrets.json (local).")
    except FileNotFoundError:
        raise SystemExit("❌ Credenciales no disponibles. Configura Databricks Secrets (scope='aws') o coloca aws_secrets.json.")
    except json.JSONDecodeError:
        raise SystemExit("❌ aws_secrets.json inválido.")

AWS_KEY = config["aws_access_key"]
AWS_SECRET = config["aws_secret_key"]
BUCKET = "bronce-scrap-date"

S3_OPTS = {
    "fs.s3a.access.key": AWS_KEY,
    "fs.s3a.secret.key": AWS_SECRET,
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

# Reglas por fuente derivadas del análisis local de raw parquet.
# Se mantienen fuera del notebook para facilitar mantenimiento y trazabilidad.
_candidates = []
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _repo_dir = "/Workspace" + str(_nb_path).rsplit("/", 1)[0]
    _candidates.append(_repo_dir)
except Exception:
    pass
_vsc_file = globals().get("__vsc_ipynb_file__", "")
if _vsc_file:
    _candidates.append(os.path.dirname(os.path.abspath(_vsc_file)))
_candidates.append(os.getcwd())
for _candidate in _candidates:
    if _candidate and _candidate not in sys.path:
        sys.path.insert(0, _candidate)

def load_repo_module(module_name):
    try:
        module = importlib.import_module(module_name)
        module = importlib.reload(module)
        return module
    except Exception as import_exc:
        last_error = import_exc
        for candidate_dir in _candidates:
            if not candidate_dir:
                continue
            module_path = os.path.join(candidate_dir, f"{module_name}.py")
            if not os.path.isfile(module_path):
                continue
            try:
                spec = importlib.util.spec_from_file_location(module_name, module_path)
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)
                sys.modules[module_name] = module
                return module
            except Exception as file_exc:
                last_error = file_exc
        raise last_error

try:
    source_quality_rules = load_repo_module("source_quality_rules")
    SOURCE_PRICE_RULES = source_quality_rules.SOURCE_PRICE_RULES
    DEFAULT_PRICE_RULES = source_quality_rules.DEFAULT_PRICE_RULES
    SOURCE_REQUIRED_SIGNALS = source_quality_rules.SOURCE_REQUIRED_SIGNALS
    print("✅ source_quality_rules.py cargado.")
except Exception as exc:
    raise SystemExit(
        "❌ No se pudo cargar source_quality_rules.py. "
        "El archivo debe existir en la raíz del repo para centralizar reglas de calidad. "
        f"Detalle: {str(exc)[:200]}"
    )

GEOGRAPHY_CATALOG_FALLBACK = {
    "bogota": {"aliases": ["bogota", "bogota d c", "bogota dc", "bogota distrito capital"], "department": "bogota_dc", "region": "andina", "market": "bogota_metropolitana"},
    "medellin": {"aliases": ["medellin"], "department": "antioquia", "region": "andina", "market": "valle_aburra"},
    "cali": {"aliases": ["cali", "santiago de cali"], "department": "valle_del_cauca", "region": "pacifica", "market": "cali_metropolitana"},
    "barranquilla": {"aliases": ["barranquilla", "bquilla"], "department": "atlantico", "region": "caribe", "market": "barranquilla_metropolitana"},
    "cartagena": {"aliases": ["cartagena", "cartagena de indias"], "department": "bolivar", "region": "caribe", "market": "cartagena_metropolitana"},
    "bucaramanga": {"aliases": ["bucaramanga"], "department": "santander", "region": "andina", "market": "bucaramanga_metropolitana"},
    "pereira": {"aliases": ["pereira"], "department": "risaralda", "region": "andina", "market": "eje_cafetero"},
    "manizales": {"aliases": ["manizales"], "department": "caldas", "region": "andina", "market": "eje_cafetero"},
    "cucuta": {"aliases": ["cucuta", "san jose de cucuta"], "department": "norte_de_santander", "region": "andina", "market": "cucuta_metropolitana"},
    "ibague": {"aliases": ["ibague"], "department": "tolima", "region": "andina", "market": "ibague_metropolitana"},
    "santa marta": {"aliases": ["santa marta", "sta marta"], "department": "magdalena", "region": "caribe", "market": "santa_marta_metropolitana"},
    "villavicencio": {"aliases": ["villavicencio"], "department": "meta", "region": "orinoquia", "market": "villavicencio_metropolitana"},
    "pasto": {"aliases": ["pasto", "san juan de pasto"], "department": "narino", "region": "pacifica", "market": "pasto_metropolitana"},
    "monteria": {"aliases": ["monteria"], "department": "cordoba", "region": "caribe", "market": "monteria_metropolitana"},
    "neiva": {"aliases": ["neiva"], "department": "huila", "region": "andina", "market": "neiva_metropolitana"},
    "armenia": {"aliases": ["armenia"], "department": "quindio", "region": "andina", "market": "eje_cafetero"},
    "popayan": {"aliases": ["popayan"], "department": "cauca", "region": "pacifica", "market": "popayan_metropolitana"},
    "valledupar": {"aliases": ["valledupar"], "department": "cesar", "region": "caribe", "market": "valledupar_metropolitana"},
    "sincelejo": {"aliases": ["sincelejo"], "department": "sucre", "region": "caribe", "market": "sincelejo_metropolitana"},
    "tunja": {"aliases": ["tunja"], "department": "boyaca", "region": "andina", "market": "tunja_metropolitana"},
    "envigado": {"aliases": ["envigado"], "department": "antioquia", "region": "andina", "market": "valle_aburra"},
    "sabaneta": {"aliases": ["sabaneta"], "department": "antioquia", "region": "andina", "market": "valle_aburra"},
    "itagui": {"aliases": ["itagui"], "department": "antioquia", "region": "andina", "market": "valle_aburra"},
    "bello": {"aliases": ["bello"], "department": "antioquia", "region": "andina", "market": "valle_aburra"},
    "soacha": {"aliases": ["soacha"], "department": "cundinamarca", "region": "andina", "market": "bogota_metropolitana"},
    "chia": {"aliases": ["chia", "chia cundinamarca"], "department": "cundinamarca", "region": "andina", "market": "bogota_metropolitana"},
    "zipaquira": {"aliases": ["zipaquira"], "department": "cundinamarca", "region": "andina", "market": "bogota_metropolitana"},
    "cajica": {"aliases": ["cajica"], "department": "cundinamarca", "region": "andina", "market": "bogota_metropolitana"},
    "cota": {"aliases": ["cota"], "department": "cundinamarca", "region": "andina", "market": "bogota_metropolitana"},
    "funza": {"aliases": ["funza"], "department": "cundinamarca", "region": "andina", "market": "bogota_metropolitana"},
    "mosquera": {"aliases": ["mosquera"], "department": "cundinamarca", "region": "andina", "market": "bogota_metropolitana"},
    "tocancipa": {"aliases": ["tocancipa"], "department": "cundinamarca", "region": "andina", "market": "bogota_metropolitana"},
    "la calera": {"aliases": ["la calera"], "department": "cundinamarca", "region": "andina", "market": "bogota_metropolitana"},
    "sopo": {"aliases": ["sopo"], "department": "cundinamarca", "region": "andina", "market": "bogota_metropolitana"},
    "rionegro": {"aliases": ["rionegro"], "department": "antioquia", "region": "andina", "market": "oriente_antioqueno"},
    "girardot": {"aliases": ["girardot"], "department": "cundinamarca", "region": "andina", "market": "bogota_metropolitana"},
    "floridablanca": {"aliases": ["floridablanca"], "department": "santander", "region": "andina", "market": "bucaramanga_metropolitana"},
    "piedecuesta": {"aliases": ["piedecuesta"], "department": "santander", "region": "andina", "market": "bucaramanga_metropolitana"},
}

def build_geography_maps(city_catalog):
    city_alias_to_canonical = {}
    city_to_department = {}
    city_to_region = {}
    city_to_market = {}
    for canonical_city, payload in city_catalog.items():
        city_alias_to_canonical[canonical_city] = canonical_city
        city_to_department[canonical_city] = payload["department"]
        city_to_region[canonical_city] = payload["region"]
        city_to_market[canonical_city] = payload.get("market", canonical_city)
        for alias in payload.get("aliases", []):
            city_alias_to_canonical[alias] = canonical_city
    sorted_city_aliases = sorted(city_alias_to_canonical.keys(), key=len, reverse=True)
    return city_alias_to_canonical, city_to_department, city_to_region, city_to_market, sorted_city_aliases

try:
    geography_catalog = load_repo_module("geography_catalog")
    CITY_ALIAS_TO_CANONICAL = geography_catalog.CITY_ALIAS_TO_CANONICAL
    CITY_TO_DEPARTMENT = geography_catalog.CITY_TO_DEPARTMENT
    CITY_TO_REGION = geography_catalog.CITY_TO_REGION
    CITY_TO_MARKET = geography_catalog.CITY_TO_MARKET
    SORTED_CITY_ALIASES = geography_catalog.SORTED_CITY_ALIASES
    print("✅ geography_catalog.py cargado.")
except Exception as exc:
    (
        CITY_ALIAS_TO_CANONICAL,
        CITY_TO_DEPARTMENT,
        CITY_TO_REGION,
        CITY_TO_MARKET,
        SORTED_CITY_ALIASES,
    ) = build_geography_maps(GEOGRAPHY_CATALOG_FALLBACK)
    print(f"⚠️ geography_catalog.py no visible en este runtime; usando fallback embebido. Detalle: {str(exc)[:160]}")

# ─────────────────────────────────────────────────────────────
# UDFs DE NORMALIZACIÓN (se aplican en Silver, antes de todo)
# ─────────────────────────────────────────────────────────────

def _normalizar_tipo(tipo_raw):
    """396 variantes libres → 6 categorías canónicas."""
    if tipo_raw is None:
        return "otro"
    t = tipo_raw.lower().strip()
    if any(x in t for x in ["apto", "apart", "piso", "estudio", "loft", "penthouse", "duplex"]):
        return "apartamento"
    if any(x in t for x in ["casa", "chalet", "villa", "finca", "cabana", "cabañ"]):
        return "casa"
    if any(x in t for x in ["oficin", "consultori"]):
        return "oficina"
    if any(x in t for x in ["local", "bodega", "comerci", "nave"]):
        return "local_comercial"
    if any(x in t for x in ["lote", "terreno", "parcela"]):
        return "lote"
    return "otro"

normalizar_tipo_udf = F.udf(_normalizar_tipo, StringType())


def _normalizar_estado(estado_raw):
    """Variantes de estado → 4 categorías."""
    if estado_raw is None:
        return "desconocido"
    t = estado_raw.lower().strip()
    if any(x in t for x in ["nuevo", "new", "estrenar"]):
        return "nuevo"
    if any(x in t for x in ["usado", "second", "segunda"]):
        return "usado"
    if any(x in t for x in ["proyecto", "plano", "sobre plano", "preventa"]):
        return "sobre_planos"
    if any(x in t for x in ["remodelado", "renovado"]):
        return "remodelado"
    return "desconocido"

normalizar_estado_udf = F.udf(_normalizar_estado, StringType())


def _normalizar_ubicacion(text):
    """Normaliza ubicación: quita acentos, lowercase, limpia separadores."""
    if text is None:
        return None
    nfkd = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in nfkd if not unicodedata.combining(c))
    text = text.lower().strip()
    for noise in ["d.c.", "d.c", "dc", "colombia", "departamento", "municipio"]:
        text = text.replace(noise, "")
    text = re.sub(r"[,\-–—/|·•()]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

normalizar_ubicacion_udf = F.udf(_normalizar_ubicacion, StringType())


def _extraer_ciudad(ubicacion_norm):
    if ubicacion_norm is None:
        return "otra_ciudad"

    text = f" {ubicacion_norm.strip()} "
    for alias in SORTED_CITY_ALIASES:
        if f" {alias} " in text:
            return CITY_ALIAS_TO_CANONICAL[alias]

    return "otra_ciudad"

extraer_ciudad_udf = F.udf(_extraer_ciudad, StringType())


def _map_city_department(city_token):
    if city_token is None:
        return "departamento_otra"
    return CITY_TO_DEPARTMENT.get(city_token, "departamento_otra")

map_city_department_udf = F.udf(_map_city_department, StringType())


def _map_city_region(city_token):
    if city_token is None:
        return "region_otra"
    return CITY_TO_REGION.get(city_token, "region_otra")

map_city_region_udf = F.udf(_map_city_region, StringType())


def _map_city_market(city_token):
    if city_token is None:
        return "mercado_otro"
    return CITY_TO_MARKET.get(city_token, "mercado_otro")

map_city_market_udf = F.udf(_map_city_market, StringType())

In [0]:
# =============================================================
# CELDA 2: FUNCIÓN — Normalización Universal Multi-Fuente
# =============================================================
# Cada portal usa nombres diferentes para el mismo concepto:
#   precio/price, ubicacion/location/city, titulo/title, url/listing_url
# Esta función usa coalesce() para mapear TODAS las variantes sin if/else.
# Además aplica normalización estricta de tipo_inmueble, estado, ubicación.

def _safe_extract_number(col_expr, pattern, cast_type="double"):
    """regexp_extract retorna '' cuando no hay match → cast falla.
    Esta helper retorna NULL en vez de '' antes del cast."""
    extracted = regexp_extract(col_expr, pattern, 1)
    return when(length(extracted) > 0, extracted.cast(cast_type))

def normalizar_fuente(fuente):
    """Lee Bronze Delta de una fuente y normaliza a schema Silver unificado.
    Retorna DataFrame limpio + métricas para control de calidad temprano."""
    ruta = f"s3a://{BUCKET}/bronze/{fuente}/"
    print(f"📖 {fuente}...")

    try:
        reader = spark.read.format("delta")
        for k, v in S3_OPTS.items():
            reader = reader.option(k, v)
        df = reader.load(ruta)
    except Exception as e:
        print(f"  ⚠️ Error leyendo {fuente}: {e}")
        return None, {"fuente": fuente, "error": str(e)[:200]}

    all_cols = set(df.columns)
    print(f"  📊 {fuente}: {len(all_cols)} columnas Bronze")

    price_rules = SOURCE_PRICE_RULES.get(fuente, DEFAULT_PRICE_RULES)
    min_price = price_rules["min_price"]
    max_price = price_rules["max_price"]

    def safe(name):
        """Retorna columna si existe, NULL si no."""
        return col(name) if name in all_cols else lit(None)

    # ═══════════════════════════════════════════════════════════
    # PASO 1: MAPEO UNIVERSAL DE COLUMNAS
    # ═══════════════════════════════════════════════════════════
    df_norm = df.select(
        coalesce(safe("id_original"), safe("id"), safe("property_id"), safe("listing_id")).alias("id_original"),

        coalesce(
            safe("fecha_extraccion"), safe("extracted_at"),
            safe("ingestion_timestamp"), safe("bronze_ingested_at")
        ).alias("fecha_extraccion"),

        coalesce(safe("source_system"), safe("portal"), safe("source")).alias("fuente"),

        coalesce(safe("url"), safe("listing_url")).alias("url"),

        coalesce(safe("titulo"), safe("title")).alias("titulo"),

        # --- PARCHE: ubicacion_raw enriquecida ---
        # Prioriza campos ricos (ubicacion completa, address) sobre city a secas.
        # Así conservamos barrio/sector cuando el portal lo incluye.
        coalesce(
            safe("ubicacion"), safe("location"),
            safe("address"), safe("direccion"), safe("municipio_texto"),
            safe("city")
        ).alias("ubicacion_raw"),

        coalesce(
            safe("precio_num").cast("long"),
            _safe_extract_number(safe("price"), r"(\d{5,})", "long")
        ).alias("precio_num"),

        coalesce(
            safe("area_m2").cast("double"),
            _safe_extract_number(safe("area"), r"(\d+\.?\d*)", "double")
        ).alias("area_m2"),

        _safe_extract_number(
            coalesce(safe("habitaciones"), safe("rooms"), safe("bedrooms")), r"(\d+)", "int"
        ).alias("habitaciones"),

        _safe_extract_number(coalesce(safe("banos"), safe("bathrooms"), safe("baths")), r"(\d+)", "int").alias("banos"),

        _safe_extract_number(coalesce(safe("garajes"), safe("parking"), safe("garages")), r"(\d+)", "int").alias("garajes"),

        coalesce(safe("tipo_inmueble"), safe("property_type"), safe("category")).alias("tipo_inmueble_raw"),

        coalesce(safe("estado_inmueble"), safe("status"), safe("condition")).alias("estado_inmueble_raw"),

        safe("source_file").alias("source_file"),
        safe("batch_id").alias("batch_id"),
    )

    # ═══════════════════════════════════════════════════════════
    # PASO 2: FILTROS DE CALIDAD (gates estrictos)
    # ═══════════════════════════════════════════════════════════
    n_raw = df_norm.count()
    n_with_price = df_norm.filter(col("precio_num").isNotNull()).count()

    df_clean = df_norm.filter(
        (col("precio_num").isNotNull()) &
        (col("precio_num") > min_price) &
        (col("precio_num") < max_price) &
        (col("id_original").isNotNull())
    )

    n_clean_price = df_clean.count()
    print(
        f"  🧪 {fuente}: precio válido {n_clean_price}/{n_raw} "
        f"(con precio parseado {n_with_price}/{n_raw}, gate {min_price:,} - {max_price:,})"
    )

    # ═══════════════════════════════════════════════════════════
    # PASO 3: NORMALIZACIÓN DE CAMPOS (la clave del fix)
    # ═══════════════════════════════════════════════════════════
    df_clean = (
        df_clean
        .withColumn("tipo_inmueble", normalizar_tipo_udf(col("tipo_inmueble_raw")))
        .drop("tipo_inmueble_raw")
        .withColumn("estado_inmueble", normalizar_estado_udf(col("estado_inmueble_raw")))
        .drop("estado_inmueble_raw")
        .withColumn(
            "ubicacion_raw",
            when(
                col("ubicacion_raw").isNull() |
                (length(trim(col("ubicacion_raw"))) < 3) |
                (lower(col("ubicacion_raw")) == "nan"),
                lit("Sin Ubicación")
            ).otherwise(trim(col("ubicacion_raw")))
        )
        .withColumn("ubicacion_norm", normalizar_ubicacion_udf(col("ubicacion_raw")))
        .withColumn("city_token", extraer_ciudad_udf(col("ubicacion_norm")))
        .withColumn("departamento_token", map_city_department_udf(col("city_token")))
        .withColumn("region_token", map_city_region_udf(col("city_token")))
        .withColumn("market_token", map_city_market_udf(col("city_token")))
        .withColumn(
            "titulo",
            when(col("titulo").isNull(), lit(None))
            .otherwise(regexp_replace(trim(col("titulo")), r"\s+", " "))
        )
        .withColumn(
            "area_m2",
            when((col("area_m2") >= 10) & (col("area_m2") <= 2000), col("area_m2"))
            .otherwise(lit(None))
        )
        .withColumn(
            "habitaciones",
            when((col("habitaciones") >= 0) & (col("habitaciones") <= 20), col("habitaciones"))
            .otherwise(lit(None))
        )
        .withColumn(
            "banos",
            when((col("banos") >= 0) & (col("banos") <= 15), col("banos"))
            .otherwise(lit(None))
        )
        .withColumn(
            "garajes",
            when((col("garajes") >= 0) & (col("garajes") <= 10), col("garajes"))
            .otherwise(lit(None))
        )
    )

    n_clean = df_clean.count()
    quality_stats = (
        df_clean.agg(
            F.count("*").alias("rows_clean"),
            F.sum(F.when(col("area_m2").isNotNull(), 1).otherwise(0)).alias("rows_with_area"),
            F.sum(F.when(col("titulo").isNotNull() & (length(trim(col("titulo"))) > 0), 1).otherwise(0)).alias("rows_with_title"),
            F.sum(F.when(col("url").isNotNull() & (length(trim(col("url"))) > 0), 1).otherwise(0)).alias("rows_with_url"),
            F.sum(F.when((col("city_token").isNotNull()) & (col("city_token") != "otra_ciudad"), 1).otherwise(0)).alias("rows_with_known_city"),
            F.sum(F.when((col("market_token").isNotNull()) & (col("market_token") != "mercado_otro"), 1).otherwise(0)).alias("rows_with_known_market"),
            # --- PARCHE: métrica de riqueza de ubicación ---
            F.sum(F.when(
                (col("ubicacion_norm").isNotNull()) &
                (F.length(col("ubicacion_norm")) > F.length(F.coalesce(col("city_token"), F.lit(""))) + 3),
                1
            ).otherwise(0)).alias("rows_with_rich_location"),
        ).collect()[0]
        if n_clean > 0 else None
    )

    quality_report = {
        "fuente": fuente,
        "raw_rows": int(n_raw),
        "rows_with_price": int(n_with_price),
        "clean_rows": int(n_clean),
        "discarded_rows": int(max(n_raw - n_clean, 0)),
        "pct_price_parse": float(round((n_with_price / n_raw) * 100, 2)) if n_raw else 0.0,
        "pct_survival": float(round((n_clean / n_raw) * 100, 2)) if n_raw else 0.0,
        "pct_area_complete": float(round((quality_stats["rows_with_area"] / n_clean) * 100, 2)) if quality_stats and n_clean else 0.0,
        "pct_title_complete": float(round((quality_stats["rows_with_title"] / n_clean) * 100, 2)) if quality_stats and n_clean else 0.0,
        "pct_url_complete": float(round((quality_stats["rows_with_url"] / n_clean) * 100, 2)) if quality_stats and n_clean else 0.0,
        "pct_known_city": float(round((quality_stats["rows_with_known_city"] / n_clean) * 100, 2)) if quality_stats and n_clean else 0.0,
        "pct_known_market": float(round((quality_stats["rows_with_known_market"] / n_clean) * 100, 2)) if quality_stats and n_clean else 0.0,
        "pct_rich_location": float(round((quality_stats["rows_with_rich_location"] / n_clean) * 100, 2)) if quality_stats and n_clean and "rows_with_rich_location" in quality_stats.asDict() else 0.0,
        "min_price_rule": int(min_price),
        "max_price_rule": int(max_price),
    }

    print(
        f"  ✅ {fuente}: {n_clean} válidos (de {n_raw} raw, {n_raw - n_clean} descartados) | "
        f"survival={quality_report['pct_survival']:.1f}% | "
        f"rich_location={quality_report['pct_rich_location']:.1f}%"
    )
    return df_clean, quality_report

In [0]:
# =============================================================
# CELDA 3: EJECUCIÓN — Descubrimiento Dinámico + Unión + Dedup
# =============================================================

BRONZE_PORTAL_CHECKPOINTS_PATH = config.get("bronze_portal_checkpoints_path") or f"s3a://{BUCKET}/bronze/portal_checkpoints_raw/"

def load_checkpoint_bronze(path):
    last_error = None
    for fmt in ["delta", "parquet"]:
        try:
            reader = spark.read.format(fmt)
            for k, v in S3_OPTS.items():
                reader = reader.option(k, v)
            return reader.load(path), fmt
        except Exception as exc:
            last_error = exc
    raise last_error

# Descubrir fuentes dinámicamente desde Bronze (NO hardcoded)
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name="us-east-1",
)

response = s3.list_objects_v2(Bucket=BUCKET, Prefix="bronze/", Delimiter="/")
fuentes = [
    p["Prefix"].replace("bronze/", "").strip("/")
    for p in response.get("CommonPrefixes", [])
    if p["Prefix"].strip("/") not in {"bronze", "bronze/portal_checkpoints_raw"}
    and p["Prefix"].replace("bronze/", "").strip("/") != "portal_checkpoints_raw"
]
print(f"🔍 Fuentes en Bronze: {fuentes}")

# Normalizar cada fuente y capturar métricas de calidad
dfs = []
quality_reports = []
for f in fuentes:
    df, quality_report = normalizar_fuente(f)
    if df is not None:
        dfs.append(df)
    if quality_report is not None:
        quality_reports.append(quality_report)

if not dfs:
    raise SystemExit("❌ No hay datos para procesar en Bronze.")

# Unión (allowMissingColumns maneja columnas que no existen en todas las fuentes)
df_union = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)
print(f"\n📊 Total unificado: {df_union.count()} registros")

# Deduplicación: conservar el registro más reciente por (id_original, fuente)
w = Window.partitionBy("id_original", "fuente").orderBy(desc("fecha_extraccion"))

df_silver = (
    df_union
    .withColumn("_rank", row_number().over(w))
    .filter(col("_rank") == 1)
    .drop("_rank")
    .withColumn("silver_processed_at", current_timestamp())
)

n_final = df_silver.count()
print(f"📉 Registros únicos: {n_final}")

if quality_reports:
    df_source_quality_report = (
        spark.createDataFrame(quality_reports)
        .withColumn(
            "quality_alert",
            when(col("pct_survival") < 40, lit("survival_bajo"))
            .when(col("pct_area_complete") < 50, lit("area_incompleta"))
            .when(col("pct_known_city") < 40, lit("ciudad_debil"))
            .when(col("pct_known_market") < 50, lit("mercado_debil"))
            .otherwise(lit("ok"))
        )
        .withColumn(
            "source_quality_score",
            F.round(
                F.greatest(
                    lit(0.0),
                    F.least(
                        lit(100.0),
                        col("pct_survival") * lit(0.40)
                        + col("pct_area_complete") * lit(0.20)
                        + col("pct_title_complete") * lit(0.10)
                        + col("pct_url_complete") * lit(0.10)
                        + col("pct_known_city") * lit(0.10)
                        + col("pct_known_market") * lit(0.10)
                    )
                ),
                1,
            )
        )
        .withColumn("silver_processed_at", current_timestamp())
        .orderBy(desc("source_quality_score"), desc("clean_rows"))
    )
else:
    df_source_quality_report = None

# ═══════════════════════════════════════════════════════════════
# DIAGNÓSTICO DE CALIDAD — verificar que las normalizaciones funcionan
# ═══════════════════════════════════════════════════════════════
print("\n🔬 DIAGNÓSTICO POST-NORMALIZACIÓN:")

# tipo_inmueble: debe ser ~6 categorías
print("\n── tipo_inmueble ──")
df_silver.groupBy("tipo_inmueble").count().orderBy(desc("count")).show(truncate=False)

# estado_inmueble: debe ser ~4 categorías
print("── estado_inmueble ──")
df_silver.groupBy("estado_inmueble").count().orderBy(desc("count")).show(truncate=False)

# city_token
print("── city_token (top 20) ──")
df_silver.groupBy("city_token").count().orderBy(desc("count")).show(20, truncate=False)
n_cities = df_silver.select("city_token").distinct().count()
print(f"   Total ciudades únicas: {n_cities}")

# market_token
print("── market_token (top 20) ──")
df_silver.groupBy("market_token").count().orderBy(desc("count")).show(20, truncate=False)
n_markets = df_silver.select("market_token").distinct().count()
print(f"   Total mercados únicos: {n_markets}")

# Diagnóstico de no reconocidas
otra_ciudad_count = df_silver.filter(col("city_token") == "otra_ciudad").count()
otra_ciudad_pct = round((otra_ciudad_count / n_final) * 100, 2) if n_final else 0.0
print(f"   Registros en otra_ciudad: {otra_ciudad_count} ({otra_ciudad_pct}%)")
if otra_ciudad_count > 0:
    print("── Top ubicaciones que caen en otra_ciudad ──")
    (
        df_silver
        .filter(col("city_token") == "otra_ciudad")
        .groupBy("ubicacion_norm")
        .count()
        .orderBy(desc("count"))
        .show(25, truncate=False)
    )

# Nulls por columna
print("── Completitud ──")
for c in ["precio_num", "area_m2", "habitaciones", "banos", "garajes", "tipo_inmueble", "ubicacion_raw", "city_token", "market_token"]:
    null_pct = df_silver.filter(col(c).isNull()).count() / n_final * 100
    print(f"   {c}: {100 - null_pct:.1f}% completo ({null_pct:.1f}% nulls)")

# Precio por portal
print("\n── Registros por portal ──")
df_silver.groupBy("fuente").count().orderBy(desc("count")).show(truncate=False)

# Reporte de calidad por fuente
if df_source_quality_report is not None:
    print("\n── Calidad por fuente ──")
    display(df_source_quality_report)
else:
    print("\nℹ️ No se generó reporte de calidad por fuente.")

# ═══════════════════════════════════════════════════════════════
# CHECKPOINTS — Silver técnico/limpio para consumo Gold
# ═══════════════════════════════════════════════════════════════
checkpoint_columns_expected = [
    ("portal", "string"),
    ("checkpoint_key", "string"),
    ("checkpoint_source_path", "string"),
    ("checkpoint_activo", "int"),
    ("checkpoint_last_page", "long"),
    ("checkpoint_total_scraped", "long"),
    ("checkpoint_updated_at", "timestamp"),
    ("checkpoint_updated_at_raw", "string"),
    ("checkpoint_read_error", "string"),
    ("ingestion_timestamp", "timestamp"),
    ("batch_id", "string"),
]

try:
    df_checkpoint_bronze, checkpoint_bronze_format = load_checkpoint_bronze(BRONZE_PORTAL_CHECKPOINTS_PATH)
    print(f"\n🛰️ Bronze checkpoints cargado ({checkpoint_bronze_format.upper()}): {BRONZE_PORTAL_CHECKPOINTS_PATH}")
except Exception as e:
    df_checkpoint_bronze = None
    print(f"\nℹ️ Bronze checkpoints no disponible: {str(e)[:200]}")

if df_checkpoint_bronze is not None:
    for col_name, col_type in checkpoint_columns_expected:
        if col_name not in df_checkpoint_bronze.columns:
            df_checkpoint_bronze = df_checkpoint_bronze.withColumn(col_name, F.lit(None).cast(col_type))

    checkpoint_window = Window.partitionBy("portal").orderBy(
        col("checkpoint_updated_at").desc_nulls_last(),
        col("ingestion_timestamp").desc_nulls_last(),
        col("checkpoint_total_scraped").desc_nulls_last(),
    )

    df_silver_checkpoints = (
        df_checkpoint_bronze
        .select([name for name, _ in checkpoint_columns_expected])
        .withColumn("portal", lower(trim(col("portal"))))
        .filter(col("portal").isNotNull() & (col("portal") != ""))
        .withColumn("checkpoint_activo", coalesce(col("checkpoint_activo"), lit(0)))
        .withColumn("checkpoint_has_error", when(col("checkpoint_read_error").isNotNull() & (length(trim(col("checkpoint_read_error"))) > 0), lit(1)).otherwise(lit(0)))
        .withColumn("checkpoint_source_system", regexp_extract(col("checkpoint_key"), r"raw/([^/]+)/", 1))
        .withColumn("checkpoint_ingested_at", coalesce(col("ingestion_timestamp"), current_timestamp()))
        .withColumn("_rank", row_number().over(checkpoint_window))
        .filter(col("_rank") == 1)
        .drop("_rank")
        .withColumn("silver_processed_at", current_timestamp())
    )

    checkpoint_count = df_silver_checkpoints.count()
    print(f"   Checkpoints Silver consolidados: {checkpoint_count}")
    df_silver_checkpoints.groupBy("portal").count().orderBy(desc("count")).show(20, truncate=False)
else:
    df_silver_checkpoints = None
    print("   Checkpoints Silver omitido en esta corrida.")

In [0]:
# =============================================================
# CELDA 4: GUARDAR MASTER SILVER EN S3
# =============================================================

SILVER_PATH = f"s3a://{BUCKET}/silver/master_inmuebles/"
SILVER_CHECKPOINTS_PATH = f"s3a://{BUCKET}/silver/portal_checkpoints/"
SILVER_SOURCE_QUALITY_PATH = f"s3a://{BUCKET}/silver/source_quality_report/"

def write_delta_table(df, path, label):
    writer = (
        df.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
    )
    for k, v in S3_OPTS.items():
        writer = writer.option(k, v)
    writer.save(path)
    print(f"✅ {label} guardado en Delta: {path}")

write_delta_table(df_silver, SILVER_PATH, "Silver")

if df_source_quality_report is not None:
    write_delta_table(
        df_source_quality_report,
        SILVER_SOURCE_QUALITY_PATH,
        "Reporte de calidad por fuente",
    )
else:
    print("ℹ️ No se generó reporte de calidad por fuente en esta corrida.")

if df_silver_checkpoints is not None:
    write_delta_table(
        df_silver_checkpoints,
        SILVER_CHECKPOINTS_PATH,
        "Checkpoints Silver",
    )
else:
    print("ℹ️ No se generó tabla Silver de checkpoints en esta corrida.")

In [0]:
# =============================================================
# CELDA 5: AUDITORÍA GEOGRÁFICA CRUDA -> EXCEL
# =============================================================
import pandas as pd
import os

def _safe_pick(df, candidates, alias_name):
    for candidate in candidates:
        if candidate in df.columns:
            return col(candidate).cast("string").alias(alias_name)
    return lit(None).cast("string").alias(alias_name)

def build_raw_geography_audit(fuente):
    path = f"s3a://{BUCKET}/bronze/{fuente}/"
    try:
        raw_df = spark.read.options(**S3_OPTS).format("delta").load(path)
    except Exception as exc:
        print(f"⚠️ No se pudo leer bronze/{fuente} para auditoría geográfica: {str(exc)[:160]}")
        return None

    geo_df = (
        raw_df
        .select(
            lit(fuente).alias("fuente"),
            _safe_pick(raw_df, ["id_original", "id", "property_id", "listing_id"], "id_original"),
            _safe_pick(raw_df, ["ubicacion", "location", "address", "direccion", "municipio_texto"], "ubicacion_raw"),
            _safe_pick(raw_df, ["ciudad", "city", "municipio", "town"], "city_raw"),
            _safe_pick(raw_df, ["departamento", "department", "state", "provincia", "province"], "department_raw"),
            _safe_pick(raw_df, ["region", "zona", "zone", "macro_region"], "region_raw"),
            _safe_pick(raw_df, ["titulo", "title", "nombre", "name"], "titulo_raw"),
        )
        .withColumn("ubicacion_raw", trim(col("ubicacion_raw")))
        .withColumn("city_raw", trim(col("city_raw")))
        .withColumn("department_raw", trim(col("department_raw")))
        .withColumn("region_raw", trim(col("region_raw")))
        .withColumn(
            "geo_texto_base",
            coalesce(
                col("ubicacion_raw"),
                F.concat_ws(" | ", col("city_raw"), col("department_raw"), col("region_raw"))
            )
        )
        .withColumn("geo_texto_norm", normalizar_ubicacion_udf(col("geo_texto_base")))
        .withColumn("city_token_inferido", extraer_ciudad_udf(col("geo_texto_norm")))
        .withColumn("departamento_token_inferido", map_city_department_udf(col("city_token_inferido")))
        .withColumn("region_token_inferido", map_city_region_udf(col("city_token_inferido")))
        .withColumn("market_token_inferido", map_city_market_udf(col("city_token_inferido")))
    )
    return geo_df

def resolve_geo_audit_output_path():
    # Flexible logic: Try workspace, dbfs if possible, else fallback to cwd
    preferred_paths = []
    # Try workspace directory
    workspace_dir = '/Workspace/Users/nicolasguarin1997@gmail.com/raw_geography_audit.xlsx'
    try:
        os.makedirs(os.path.dirname(workspace_dir), exist_ok=True)
        preferred_paths.append(workspace_dir)
    except Exception:
        pass
    # Try DBFS and tmp if dbutils available
    if "dbutils" in globals():
        preferred_paths.extend([
            '/dbfs/FileStore/lakehouse_audits/raw_geography_audit.xlsx',
            '/tmp/lakehouse_audits/raw_geography_audit.xlsx',
        ])
    # Always fallback to cwd for reliability
    preferred_paths.append(os.path.join(os.getcwd(), 'raw_geography_audit.xlsx'))
    last_error = None
    for candidate in preferred_paths:
        try:
            parent_dir = os.path.dirname(candidate) or os.getcwd()
            os.makedirs(parent_dir, exist_ok=True)
            return candidate
        except Exception as exc:
            last_error = exc
            continue
    raise SystemExit(
        "❌ No fue posible resolver una ruta de salida para el Excel de auditoría geográfica. "
        f"Detalle: {str(last_error)[:180] if last_error else 'sin detalle'}"
    )

audit_frames = []
for fuente in fuentes:
    audit_df = build_raw_geography_audit(fuente)
    if audit_df is not None:
        audit_frames.append(audit_df)

if not audit_frames:
    print("⚠️ No se pudo construir ninguna auditoría geográfica desde Bronze.")
else:
    df_raw_geo_audit = reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), audit_frames)

    resumen_fuente = (
        df_raw_geo_audit
        .groupBy(
            "fuente",
            "city_token_inferido",
            "departamento_token_inferido",
            "region_token_inferido",
            "market_token_inferido",
        )
        .agg(F.count("*").alias("n_registros"))
        .orderBy(desc("n_registros"))
        .limit(1000)
        .toPandas()
    )

    mercados_top = (
        df_raw_geo_audit
        .groupBy("fuente", "market_token_inferido")
        .agg(F.count("*").alias("n_registros"))
        .orderBy(desc("n_registros"))
        .limit(1000)
        .toPandas()
    )

    ciudades_crudas_top = (
        df_raw_geo_audit
        .withColumn("city_raw_norm", normalizar_ubicacion_udf(col("city_raw")))
        .groupBy("fuente", "city_raw_norm")
        .agg(F.count("*").alias("n_registros"))
        .orderBy(desc("n_registros"))
        .limit(1000)
        .toPandas()
    )

    departamentos_crudos_top = (
        df_raw_geo_audit
        .withColumn("department_raw_norm", normalizar_ubicacion_udf(col("department_raw")))
        .groupBy("fuente", "department_raw_norm")
        .agg(F.count("*").alias("n_registros"))
        .orderBy(desc("n_registros"))
        .limit(1000)
        .toPandas()
    )

    regiones_crudas_top = (
        df_raw_geo_audit
        .withColumn("region_raw_norm", normalizar_ubicacion_udf(col("region_raw")))
        .groupBy("fuente", "region_raw_norm")
        .agg(F.count("*").alias("n_registros"))
        .orderBy(desc("n_registros"))
        .limit(1000)
        .toPandas()
    )

    otra_ciudad_review = (
        df_raw_geo_audit
        .filter(col("city_token_inferido") == "otra_ciudad")
        .groupBy(
            "fuente",
            "geo_texto_norm",
            "city_raw",
            "department_raw",
            "region_raw",
        )
        .agg(F.count("*").alias("n_registros"))
        .orderBy(desc("n_registros"))
        .limit(1500)
        .toPandas()
    )

    sugerencia_mapeo = (
        df_raw_geo_audit
        .filter(col("city_token_inferido") == "otra_ciudad")
        .groupBy("fuente", "geo_texto_norm")
        .agg(F.count("*").alias("n_registros"))
        .orderBy(desc("n_registros"))
        .limit(500)
        .toPandas()
    )
    sugerencia_mapeo["accion_sugerida"] = "revisar_alias_o_submercado"
    sugerencia_mapeo["city_token_sugerido"] = ""
    sugerencia_mapeo["market_token_sugerido"] = ""
    sugerencia_mapeo["comentario"] = "Priorizar por volumen y decidir si es municipio, submercado o ruido"

    muestra_detalle = (
        df_raw_geo_audit
        .select(
            "fuente",
            "id_original",
            "ubicacion_raw",
            "city_raw",
            "department_raw",
            "region_raw",
            "geo_texto_norm",
            "city_token_inferido",
            "departamento_token_inferido",
            "region_token_inferido",
            "market_token_inferido",
            "titulo_raw",
        )
        .limit(2000)
        .toPandas()
    )

    output_path = resolve_geo_audit_output_path()
    engine_options = [None, "openpyxl", "xlsxwriter"]
    last_error = None

    for engine in engine_options:
        try:
            writer_kwargs = {"engine": engine} if engine else {}
            with pd.ExcelWriter(output_path, **writer_kwargs) as writer:
                resumen_fuente.to_excel(writer, sheet_name="resumen_fuente", index=False)
                mercados_top.to_excel(writer, sheet_name="market_top", index=False)
                ciudades_crudas_top.to_excel(writer, sheet_name="city_raw_top", index=False)
                departamentos_crudos_top.to_excel(writer, sheet_name="department_raw_top", index=False)
                regiones_crudas_top.to_excel(writer, sheet_name="region_raw_top", index=False)
                otra_ciudad_review.to_excel(writer, sheet_name="otra_ciudad_review", index=False)
                sugerencia_mapeo.to_excel(writer, sheet_name="sugerencia_mapeo", index=False)
                muestra_detalle.to_excel(writer, sheet_name="muestra_detalle", index=False)
            last_error = None
            break
        except ModuleNotFoundError as exc:
            last_error = exc
            continue
        except Exception as exc:
            last_error = exc
            break

    if last_error is not None:
        raise SystemExit(
            "❌ No se pudo escribir el Excel de auditoría geográfica. "
            "Instala openpyxl o xlsxwriter en el runtime si hace falta. "
            f"Detalle: {str(last_error)[:200]}"
        )

    print(f"✅ Auditoría geográfica exportada a: {output_path}")
    if output_path.startswith("/dbfs/"):
        print(f"📎 Leer con pandas desde: {output_path}")
        print(f"📎 Descargar o abrir desde DBFS: dbfs:{output_path[5:]}")
    print("Hojas generadas: resumen_fuente, market_top, city_raw_top, department_raw_top, region_raw_top, otra_ciudad_review, sugerencia_mapeo, muestra_detalle")